In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_score,
    recall_score,
    roc_auc_score,
)

# 1. Đọc dữ liệu train.csv (đã có sẵn cột fold được chia từ StratifiedKFold)
print("Đang đọc dữ liệu train.csv...")
train = pd.read_csv("train_with_anomaly_score.csv")

target_col = "is_fraud"
id_col = "id"
fold_col = "fold"

# Loại trừ id, target và fold khỏi tập đặc trưng huấn luyện
feature_cols = [c for c in train.columns if c not in [target_col, id_col, fold_col]]

# 2. Xử lý missing và ép kiểu str cho các cột categorical để CatBoost xử lý trực tiếp
cat_cols = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
for col in cat_cols:
    train[col] = train[col].fillna("Missing").astype(str)

print(f"Số lượng đặc trưng (features): {len(feature_cols)}")
print(f"Danh sách các folds: {sorted(train[fold_col].unique())}")

Đang đọc dữ liệu train.csv...
Số lượng đặc trưng (features): 25
Danh sách các folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [4]:
# 3. Huấn luyện 5 model CatBoost theo K-Fold
# - Không sử dụng scale_pos_weight / auto_class_weights
# - Sử dụng eval_metric='AUC' (ROC-AUC)

models = []
oof_preds = np.zeros(len(train))
fold_metrics = []

n_splits = train[fold_col].nunique()
print(f"Bắt đầu huấn luyện {n_splits}-Fold CatBoost với eval_metric='AUC'...")

for fold in range(n_splits):
    print(f"\n{'='*20} FOLD {fold} {'='*20}")
    
    train_data = train[train[fold_col] != fold]
    val_data = train[train[fold_col] == fold]
    
    X_train, y_train = train_data[feature_cols], train_data[target_col].astype(int)
    X_val, y_val = val_data[feature_cols], val_data[target_col].astype(int)
    val_indices = val_data.index
    
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()  # Tính scale_pos_weight cho fold hiện tại
    
    model = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        eval_metric="AUC",     
        #auto_class_weights="Balanced",  # Sử dụng auto_class_weights để cân bằng lớp
        task_type="GPU",              # Huấn luyện trên GPU NVIDIA RTX 3050
        random_seed=42 + fold,
        early_stopping_rounds=100,
        verbose=100
    )
    
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    models.append(model)
    
    # Dự đoán xác suất trên tập validation của fold
    val_prob = model.predict_proba(val_pool)[:, 1]
    oof_preds[val_indices] = val_prob
    
    # Tính 4 metric cho fold này
    val_pred = (val_prob >= 0.5).astype(int)
    f_roc_auc = roc_auc_score(y_val, val_prob)
    f_auc_pr = average_precision_score(y_val, val_prob)
    f_prec = precision_score(y_val, val_pred, zero_division=0)
    f_rec = recall_score(y_val, val_pred, zero_division=0)
    
    fold_metrics.append({
        "fold": fold,
        "ROC-AUC": f_roc_auc,
        "AUC-PR": f_auc_pr,
        "Precision": f_prec,
        "Recall": f_rec
    })
    print(f"Fold {fold} - ROC-AUC: {f_roc_auc:.4f} | AUC-PR: {f_auc_pr:.4f} | Precision: {f_prec:.4f} | Recall: {f_rec:.4f}")

Bắt đầu huấn luyện 5-Fold CatBoost với eval_metric='AUC'...

==================== FOLD 0 ====================


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8605738	best: 0.8605738 (0)	total: 200ms	remaining: 4m 59s
100:	test: 0.9382677	best: 0.9382677 (100)	total: 7.77s	remaining: 1m 47s
200:	test: 0.9409533	best: 0.9409533 (200)	total: 15.4s	remaining: 1m 39s
300:	test: 0.9420193	best: 0.9420214 (299)	total: 23.2s	remaining: 1m 32s
400:	test: 0.9425893	best: 0.9425893 (400)	total: 30.7s	remaining: 1m 24s
500:	test: 0.9429963	best: 0.9429963 (500)	total: 38.8s	remaining: 1m 17s
600:	test: 0.9433152	best: 0.9433152 (600)	total: 47.1s	remaining: 1m 10s
700:	test: 0.9435969	best: 0.9436051 (684)	total: 55.4s	remaining: 1m 3s
800:	test: 0.9436383	best: 0.9436675 (779)	total: 1m 3s	remaining: 55.2s
bestTest = 0.943667531
bestIteration = 779
Shrink model to first 780 iterations.
Fold 0 - ROC-AUC: 0.9437 | AUC-PR: 0.4944 | Precision: 0.7542 | Recall: 0.2875

==================== FOLD 1 ====================


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8471768	best: 0.8471768 (0)	total: 129ms	remaining: 3m 12s
100:	test: 0.9398387	best: 0.9398387 (100)	total: 7.65s	remaining: 1m 46s
200:	test: 0.9431190	best: 0.9431190 (200)	total: 15.2s	remaining: 1m 38s
300:	test: 0.9446132	best: 0.9446132 (300)	total: 22.9s	remaining: 1m 31s
400:	test: 0.9452789	best: 0.9452789 (400)	total: 30.4s	remaining: 1m 23s
500:	test: 0.9457436	best: 0.9457436 (500)	total: 38.4s	remaining: 1m 16s
600:	test: 0.9460993	best: 0.9460998 (599)	total: 46.1s	remaining: 1m 8s
700:	test: 0.9463482	best: 0.9463482 (700)	total: 54s	remaining: 1m 1s
800:	test: 0.9466248	best: 0.9466271 (799)	total: 1m 1s	remaining: 53.9s
900:	test: 0.9467614	best: 0.9467704 (849)	total: 1m 9s	remaining: 46s
1000:	test: 0.9467896	best: 0.9468056 (967)	total: 1m 16s	remaining: 38.4s
1100:	test: 0.9468362	best: 0.9468362 (1100)	total: 1m 24s	remaining: 30.7s
1200:	test: 0.9468946	best: 0.9469042 (1199)	total: 1m 33s	remaining: 23.2s
bestTest = 0.9469041824
bestIteration = 1199


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8632999	best: 0.8632999 (0)	total: 75.6ms	remaining: 1m 53s
100:	test: 0.9445477	best: 0.9445477 (100)	total: 7.51s	remaining: 1m 44s
200:	test: 0.9478993	best: 0.9478993 (200)	total: 14.9s	remaining: 1m 36s
300:	test: 0.9490193	best: 0.9490193 (300)	total: 22.4s	remaining: 1m 29s
400:	test: 0.9494194	best: 0.9494194 (400)	total: 30.1s	remaining: 1m 22s
500:	test: 0.9497815	best: 0.9497830 (498)	total: 37.9s	remaining: 1m 15s
600:	test: 0.9499030	best: 0.9499131 (598)	total: 45.4s	remaining: 1m 7s
700:	test: 0.9500755	best: 0.9500852 (686)	total: 53.2s	remaining: 1m
800:	test: 0.9501421	best: 0.9501441 (799)	total: 1m	remaining: 53.2s
900:	test: 0.9502437	best: 0.9502437 (900)	total: 1m 8s	remaining: 45.7s
1000:	test: 0.9502230	best: 0.9502802 (959)	total: 1m 16s	remaining: 38.1s
bestTest = 0.9502801597
bestIteration = 959
Shrink model to first 960 iterations.
Fold 2 - ROC-AUC: 0.9503 | AUC-PR: 0.5197 | Precision: 0.7885 | Recall: 0.3109

==================== FOLD 3 ========

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8342104	best: 0.8342104 (0)	total: 81.5ms	remaining: 2m 2s
100:	test: 0.9405820	best: 0.9405820 (100)	total: 7.77s	remaining: 1m 47s
200:	test: 0.9434245	best: 0.9434249 (199)	total: 15.7s	remaining: 1m 41s
300:	test: 0.9441137	best: 0.9441137 (300)	total: 23.2s	remaining: 1m 32s
400:	test: 0.9443874	best: 0.9443982 (396)	total: 30.7s	remaining: 1m 24s
500:	test: 0.9446183	best: 0.9446259 (497)	total: 39.5s	remaining: 1m 18s
600:	test: 0.9448156	best: 0.9448227 (589)	total: 48.3s	remaining: 1m 12s
700:	test: 0.9448671	best: 0.9448866 (663)	total: 57.1s	remaining: 1m 5s
800:	test: 0.9448859	best: 0.9449137 (764)	total: 1m 5s	remaining: 57.4s
bestTest = 0.9449136853
bestIteration = 764
Shrink model to first 765 iterations.
Fold 3 - ROC-AUC: 0.9449 | AUC-PR: 0.5054 | Precision: 0.7570 | Recall: 0.2998

==================== FOLD 4 ====================


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8197092	best: 0.8197092 (0)	total: 324ms	remaining: 8m 5s
100:	test: 0.9347683	best: 0.9347683 (100)	total: 7.95s	remaining: 1m 50s
200:	test: 0.9380701	best: 0.9380701 (200)	total: 15.3s	remaining: 1m 38s
300:	test: 0.9390933	best: 0.9390933 (300)	total: 22.6s	remaining: 1m 30s
400:	test: 0.9397317	best: 0.9397417 (398)	total: 30.1s	remaining: 1m 22s
500:	test: 0.9400730	best: 0.9400730 (500)	total: 37.4s	remaining: 1m 14s
600:	test: 0.9403847	best: 0.9403847 (600)	total: 44.7s	remaining: 1m 6s
700:	test: 0.9406438	best: 0.9406438 (700)	total: 52s	remaining: 59.3s
800:	test: 0.9407759	best: 0.9407763 (794)	total: 59.9s	remaining: 52.3s
900:	test: 0.9408726	best: 0.9408898 (893)	total: 1m 8s	remaining: 45.5s
1000:	test: 0.9409876	best: 0.9409876 (1000)	total: 1m 16s	remaining: 38.2s
1100:	test: 0.9411539	best: 0.9411568 (1097)	total: 1m 24s	remaining: 30.6s
1200:	test: 0.9411265	best: 0.9411661 (1160)	total: 1m 35s	remaining: 23.7s
bestTest = 0.9411661029
bestIteration = 116

In [5]:
# 4. Đánh giá tổng hợp trên toàn bộ Out-Of-Fold (OOF)
y_all = train[target_col].astype(int)
oof_pred_binary = (oof_preds >= 0.5).astype(int)

oof_roc_auc = roc_auc_score(y_all, oof_preds)
oof_auc_pr = average_precision_score(y_all, oof_preds)
oof_precision = precision_score(y_all, oof_pred_binary, zero_division=0)
oof_recall = recall_score(y_all, oof_pred_binary, zero_division=0)

print("\n" + "="*45)
print("--- KẾT QUẢ TỪNG FOLD ---")
print(pd.DataFrame(fold_metrics).to_string(index=False))

print("\n" + "="*45)
print("--- TỔNG KẾT ĐÁNH GIÁ OUT-OF-FOLD (OOF) ---")
print(f"OOF ROC-AUC   : {oof_roc_auc:.4f}")
print(f"OOF AUC-PR    : {oof_auc_pr:.4f}")
print(f"OOF Precision : {oof_precision:.4f} (với threshold = 0.5)")
print(f"OOF Recall    : {oof_recall:.4f} (với threshold = 0.5)")
print("="*45)

print("\nBáo cáo chi tiết Classification Report OOF:")
print(classification_report(y_all, oof_pred_binary, digits=4))


--- KẾT QUẢ TỪNG FOLD ---
 fold  ROC-AUC   AUC-PR  Precision   Recall
    0 0.943667 0.494410   0.754209 0.287548
    1 0.946904 0.507365   0.747692 0.311805
    2 0.950280 0.519739   0.788503 0.310950
    3 0.944914 0.505351   0.757019 0.299829
    4 0.941166 0.492121   0.754777 0.304106

--- TỔNG KẾT ĐÁNH GIÁ OUT-OF-FOLD (OOF) ---
OOF ROC-AUC   : 0.9453
OOF AUC-PR    : 0.5035
OOF Precision : 0.7603 (với threshold = 0.5)
OOF Recall    : 0.3028 (với threshold = 0.5)

Báo cáo chi tiết Classification Report OOF:
              precision    recall  f1-score   support

           0     0.9883    0.9984    0.9933    688271
           1     0.7603    0.3028    0.4332     11689

    accuracy                         0.9868    699960
   macro avg     0.8743    0.6506    0.7132    699960
weighted avg     0.9845    0.9868    0.9839    699960



In [6]:
# 5. Dự đoán trên test.csv bằng cách lấy trung bình (Ensemble Averaging) từ 5 mô hình
try:
    print("Đang đọc dữ liệu test.csv...")
    test = pd.read_csv("test.csv")
    X_test = test[feature_cols].copy()
    for col in cat_cols:
        X_test[col] = X_test[col].fillna("Missing").astype(str)
    
    test_pool = Pool(X_test, cat_features=cat_cols)
    
    print("Đang dự đoán kết quả test từ 5 models...")
    test_preds = np.zeros(len(test))
    for i, model in enumerate(models):
        fold_pred = model.predict_proba(test_pool)[:, 1]
        test_preds += fold_pred / len(models)
    
    submission = pd.DataFrame({
        id_col: test[id_col],
        target_col: test_preds
    })
    submission.to_csv("submission_catboost_kfold_istree.csv", index=False)
    print("Đã lưu kết quả dự đoán vào submission_catboost_kfold.csv thành công!")
except Exception as e:
    print("Lỗi khi dự đoán test:", e)

Đang đọc dữ liệu test.csv...
Lỗi khi dự đoán test: "['anomaly_score_eif'] not in index"


In [5]:
# 6. Lưu các mô hình CatBoost thành file trọng số dạng text (.txt)
for i, model in enumerate(models):
    model_filename = f"catboost_model_fold{i}.cbm"
    # Định dạng 'text' hoặc format cbm/json/cpp/python (mặc định text format lưu cấu trúc cây và trọng số)
    model.save_model(model_filename)
    print(f"Đã lưu mô hình Fold {i} vào: {model_filename}")

print("Hoàn tất lưu toàn bộ 5 models thành file .txt!")

Đã lưu mô hình Fold 0 vào: catboost_model_fold0.cbm
Đã lưu mô hình Fold 1 vào: catboost_model_fold1.cbm
Đã lưu mô hình Fold 2 vào: catboost_model_fold2.cbm
Đã lưu mô hình Fold 3 vào: catboost_model_fold3.cbm
Đã lưu mô hình Fold 4 vào: catboost_model_fold4.cbm
Hoàn tất lưu toàn bộ 5 models thành file .txt!
